# Week 1: First Steps in Python

## Learning objectives

By the end of this session you be able to:

- Run a Python cell inside an ArcGIS Pro notebook and read what it prints
- Create variables with clear, descriptive, `snake_case` names and check their types with `type()`
- Build text output with f-strings
- Read a `NameError` and a `TypeError` traceback

## Why Python?

- “Simple”
- FOSS (free, open-source software)
- Cross-platform
- Functional
- Object-oriented
- ***Integrates with ArcGIS Pro***


## Consider the scenario
- You have a feature class containing one or more polylines. 
- **Your task:** create new points placed along each polyline at regularly spaced intervals
- Why might we want to do this? Any ideas?


<img
  src="../slide_decks/images/intro_lines.jpg"
  alt="point on a line feature"
  width="750"
/>

# Let’s break it down… how will we do it?

* Read the geometry of each feature 
* Create new points 
* Save them

<div class="spacer-sm"></div>

### Anything else we need?

<div class="spacer-lg"></div>

## One example

<img
  src="../slide_decks/images/intro_as_code.jpg"
  alt="code example of putting points along a line"
  width="550"
/>


## Or another example:

In [ ]:
# This code will not run, as the variable `lines` is not defined in this snippet. It is assumed to be a GeoDataFrame containing line geometries.
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely


def points_along_lines(gdf, spacing, include_end=False):
    """Points at fixed spacing (CRS units) along each line in gdf."""
    if gdf.crs is None or gdf.crs.is_geographic:
        raise ValueError("Requires a projected CRS; length must be in linear units.")

    parts = []
    for idx, geom in gdf.geometry.items():
        if geom is None or geom.is_empty:
            continue

        length = geom.length
        dists = np.arange(0, length, spacing)
        if include_end and not np.isclose(dists[-1], length):
            dists = np.append(dists, length)

        parts.append(
            gpd.GeoDataFrame(
                {
                    "src_index": idx,
                    "point_id": np.arange(len(dists)),
                    "chainage": dists,
                    "frac": dists / length,
                },
                geometry=shapely.line_interpolate_point(geom, dists),
                crs=gdf.crs,
            )
        )

    return gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=gdf.crs)


pts = points_along_lines(lines, spacing=25, include_end=True)
pts = pts.merge(lines.drop(columns="geometry"), left_on="src_index", right_index=True)

# Moving on...

## Opening a notebook inside ArcGIS Pro


1. Open ArcGIS Pro and start a new project
2. In the **Insert** tab, choose **New Notebook** (or right-click a folder in the **Catalog** pane and choose **New → Notebook**)
3. A notebook is built out of **cells**. Some cells hold text and formatting (like this one). Some cells hold code (like the one just below this).
4. Run a cell with **Shift+Enter**, or by clicking the ▷ button that appears next to it.
5. Cells run top to bottom, but you can re-run any single cell at any time. If your notebook starts behaving strangely, use **Restart & Run All** to get back to a clean, predictable state. The order of execution matters!

## Your first line of code

`print()` is Python's way of saying something "out loud". Whatever you put inside the parentheses gets displayed right below the cell when you run it. Text should be placed inside of quotes (we'll get to why later)

**Predict before you run:** What do you think the cell below will show once you run it? Take a guess, then run it and check.

In [ ]:
print("hello, maps")

**Predict before you run**: Below is a different use of the print() function. What do you expect will happen when you run the cell?

In [ ]:
print("Ohio has", 88, "counties.")
print("Portage", "Summit", "Stark")

Notice `print()` didn't complain or provide an error when mixing a number and text in the first line

**Your turn:** In the Python cell below, use one `print()` call to print your first name, your favorite number, and the county or city you're in right now. Use three comma-separated values - no quotes are needed around the number

In [ ]:
# Try it here


## Variables and assignment

### What is a variable?
- From algebra: a letter can represent any number, like in ```x + 2```
- In computer science, variables represent *values* or *objects* we want the computer to store in its memory for later use

- Variables don't only represent numbers
  - ...but also text and Boolean values (‘true’ or ‘false’)
  - input from the program’s user
  - to store values returned from another program
  - to represent constants
  - etc.


### Why use variables?

- Variables make your code readable and flexible
- Hard-coded values (also called "magic numbers"), mean your code is useful only in one particular scenario
- You can manually change the values in your code to fit a different scenario
  - tedious
  - greater risk of making a mistake
- Variables allow your code to be useful in many scenarios and are easy to *parameterize*, meaning you can let users change the values


You create one by writing a name, an equals sign, and the value you want to store. The `=` here doesn't mean "equals" the way it does in math. It means "assignment", or "store this value under this name."

Text values (**strings**) go inside quotes. Numbers don't need quotes.

In [ ]:
county_name = "Portage"
year = 2026

print(county_name)
print(year)

Notice `county_name` kept its quotes when we wrote it, but `year` didn't. That is a hint that Python treats these as different *kinds* of things. 

Some more examples:

In [ ]:
precip_mm = 971.4
park_acres = 450.5

print(precip_mm)
print(park_acres)

park_acres = 500.0
print(park_acres)

Something worth noticing: reassigning `park_acres` on the second-to-last line didn't create a new variable. Instead, it overwrote the box's contents. The old value, 450.5, is thrown away once the new assignment runs.

**Your turn:** In the cell below, create a variable holding a number you care about (e.g., a distance, a count, a temperature). Print it, reassign it to a different number, and print it again. Confirm for yourself that the second print shows the new value, not the old one.


In [ ]:
# Try it here


### What *else* can variables represent?

- number and string variables we worked with above represent data types that are built into Python
- Variables can also represent other things:
  - such as GIS datasets
  - tables
  - rows
  - a geoprocessor that can run tools
- All of these things are objects that you use when you work with ArcGIS in Python

## Types: or, what kind of thing is that thing?

Every value in Python has a **type**. Four you'll use often:

| Type | Example | What it holds |
|---|---|---|
| `int` | `2026` | whole numbers |
| `float` | `3.14` | numbers with a decimal point |
| `str` | `"Portage"` | text (short for "string") |
| `bool` | `True` / `False` | yes/no, on/off |

The `type()` function tells you which one you've got.

**Predict before you run:** Four values are about to go through the `type()` function: a whole number, a number with a decimal, some text, and a `True`/`False`. Guess what each one reports before you run the cell.

In [ ]:
print(type(year))
print(type(3.14))
print(type(county_name))
print(type(True))

No surprises, hopefully. We see  `int`, `float`, `str`, `bool`, in that order. 

Some additional examples:

In [ ]:
trail_count = 12
print(type(trail_count))
print(type(precip_mm))
print(type("Kent State University"))
print(type(False))

You should get the same four answers as before. `trail_count` is a brand-new variable defined here, but it's still an `int`, so `int`, `float`, `str`, `bool` all show up again, just with different values this time. 

**Your turn:** In the cell below, pick three values of your own — one whole number, one decimal number, one bit of text — assign each to a sensibly-named variable, then print each variable's type. Guess before you check.

In [ ]:
# Try it here



## Strings: concatenation vs. f-strings

At some point, you'll likely want to build a sentence out of variables. The old-school way is **concatenation**, or gluing strings together with the `+` operator. It works, but it can be a bit fussy. How? Every non-string value has to be manually converted with `str()` first, or Python will complain about mixing types.

In [ ]:
message = "Welcome to " + county_name + " County, " + str(year)
print(message)

That `str(year)` is annoying and easy to forget. **F-strings** fix this. Place an `f` right before the opening quote, and drop variables directly into the text inside curly braces. No manual conversion is then needed.

**Predict before you run:** Same message, written as an f-string this time. What do you expect to print? How do think it works?

In [ ]:
message = f"Welcome to {county_name} County, {year}"
print(message)

You get the same result, with less work, and it's easier to read. **Use f-strings as your default** for building output text this semester.

In [ ]:
precip_report = f"{county_name} County logged {precip_mm} mm of precipitation in 2025."
print(precip_report)

### How does this work?

We provided two variables of two different types (`str` and `float`). Then we used the f-string to piece them together. You can drop as many variables as you want into `{}` placeholders and let Python handle the conversion.


### What might be another advantage to building statements using f-strings?


**Your turn:** In the cell below, use `park_acres` and `county_name` to write one f-string sentence reporting a park's acreage in that county, and print it. Then rewrite the same sentence using `+` concatenation instead - which do you prefer?


In [ ]:
# Try it here


## Numbers and arithmetic

The usual operators work as you'd expect, but there are two two worth calling out: 

- `/` always returns a `float` back, even when the numbers divide evenly
- `//` (floor division) throws away the remainder and gives you a whole number

**Predict before you run:** Python has two division operators. Guess what `7 / 2` gives you, and what `7 // 2` gives you. Are they the same?

In [ ]:
print(7 / 2)
print(7 // 2)
print(7 % 2)
print(2 ** 3)

How do `%` and `**` work?

In [ ]:
tinkers_creek_acres = 450.5
wingfoot_lake_acres = 96.6
total_acres = tinkers_creek_acres + wingfoot_lake_acres
print(total_acres)

park_count_text = "88"
park_count = int(park_count_text)
print(type(park_count_text))
print(type(park_count))
print(park_count + 12)

`total_acres` was returned as `float`, the same as the two values that made it. 

The second half illustrates a different problem: `park_count_text` looks like a number, but it's actually a `str`. Python won't do arithmetic on it until you convert it explicitly with `int()`. 

Why is this relevant? Data read from a file, a form, or a table cell arrives this way constantly; `float()` does the same job when the text has a decimal point

**Your turn:** In the cell below, store the text `"500"` in a variable. Then convert it to an `int` with `int()`. Add it to `park_count`. Print the result and check its type - confirm it's an `int`, not a `str`.

In [ ]:
# Try it here


## Naming rules: snake_case and descriptive names

Python requires a few things when you name a variable:

- Names can't start with a digit or contain spaces.
- Names are case-sensitive — `year` and `Year` are two different variables.
- You *can* name a variable almost anything that follows those rules. But you *should* name it something a stranger could understand six months from now

This course's style is **`snake_case`**: lowercase words separated by underscores. For example, `county_name`, `buffer_distance_miles`, `park_count`. Not `CountyName`, not `countyname`, not `x`. Descriptive is preferred over short. `buffer_distance_miles` tells you what it is and what unit it's in; `d` tells you little/nothing.

In [ ]:
# Examples

year = 2026
Year = 2027

print(year)
print(Year)

buffer_distance_miles = 1.5
d = 1.5

print(buffer_distance_miles)
print(d)

Two things to notice above. 

- First: `year` and `Year` are two completely different variables as far as Python is concerned. A stray capital letter is a real, easy-to-miss bug
- Second: `buffer_distance_miles` and `d` print the exact same value. Python obviously doesn't care which name you chose, but be kind to "future you". Be descriptive, and you won't have to guess what `d` meant when you open the notebook 3 weeks from now.

**Your turn:** In the cell below, rewrite the line `d = 1.5` using a descriptive `snake_case` name that says what `1.5` actually measures (pick your own scenario... distance, price, whatever). Then predict: would `1st_park_acres` be a legal variable name? Why or why not?

In [ ]:
# Try it here

## Reading an error message

You will encounter a lot of errors this semester. This is a normal part of the learning process, and it's Python telling you specifically what went wrong and where. Learning to read a **traceback** (i.e., the block of text Python prints when something fails) is one of the most useful skills in this course.

The rule of thumb: **read a traceback from the bottom up.** The very last line names the error type and gives you a plain-language message. The lines above it show you the file/cell and line number where things broke.

**Predict before you run:** The cell below refers to a variable that doesn't exist (it's a typo of `county_name`). What do you think Python will do: silently ignore it, guess what you meant, or something else?

In [ ]:
print(county_nmae)

That's a **`NameError`**. Read the last line: `NameError: name 'county_nmae' is not defined`. Python looked for a variable called `county_nmae`, found nothing by that name, and told you exactly that. It didn't guess, and it didn't fail silently. A **`NameError`** fix is almost always a typo, like this one. Check spelling and capitalization first.

**Predict before you run:** This cell tries to glue a plain number directly onto a string with `+`, with no `str()` conversion. What do you think happens?

In [ ]:
print("The year is " + year)

That's a **`TypeError`**. Read the last line: `TypeError: can only concatenate str (not "int") to str`. Python is explicitly telling you which two types it refused to combine and why. Here, the `+` between strings glues text, but `year` is an `int`, so Python won't silently guess whether you meant to convert it. 

Two potential fixes: wrap it in `str(year)`, or even better (and using what you just learned) use an f-string: `f"The year is {year}"`.

In [ ]:
print(County_name)

Another `NameError`. But here, it's a different flavor than the first one. `county_nmae` was a misspelling; `County_name` is a case mismatch, exactly the `year`/`Year` example from the naming section, except this time it's a genuine bug, because `County_name` was never assigned at all. Read the last line the same way you did before: `NameError: name 'County_name' is not defined`.

**Your turn:** In the cell below, deliberately write a line of code that raises a `NameError` using a case-mismatch of a variable you've already defined (like `County_name` above, but pick a different one of your own variables). Run it, read the traceback bottom-up, then fix it back to the correctly-cased name.

In [ ]:
# Try it here

## Try it yourself

Make a variable for your hometown (a `str`) and another for its population (an `int`). Print a one-line report combining both with a single f-string. Then write a second print statement that deliberately misspells one of the variable names, run it, and read the traceback bottom-up before fixing it back.



In [ ]:
# Try it here